# Assignment 01: Linear Regression from First Principles (100 points)

**Unit**: ML1 Supervised Learning (AI 300)  
**Topics**: Normal equation, gradient descent, MSE loss, projection onto column space

---

## Background

Given a dataset of $n$ samples with $d$ features, we represent the data as a **design matrix** $X \in \mathbb{R}^{n \times d}$ and a **target vector** $y \in \mathbb{R}^n$. Linear regression finds a weight vector $w \in \mathbb{R}^d$ that minimizes the **mean squared error** (MSE):

$$\mathcal{L}(w) = \frac{1}{n}\|Xw - y\|_2^2$$

### Notation

| Symbol | Shape | Description |
|--------|-------|-------------|
| $X$ | $(n, d)$ | Design matrix (rows are samples) |
| $y$ | $(n,)$ | Target vector |
| $w$ | $(d,)$ | Weight vector |
| $\hat{y}$ | $(n,)$ | Predictions $Xw$ |
| $H$ | $(n, n)$ | Hat matrix $X(X^TX)^{-1}X^T$ |

We use the **bias trick**: prepend a column of ones to $X$ so that $w$ absorbs the bias term $b$.

In [ ]:
"""
DO NOT MAKE ANY CHANGE IN THIS CELL.
"""
import numpy as np
import matplotlib.pyplot as plt
np.random.seed(42)

> **WARNING !!!**
>
> - Beyond importing libraries/modules/classes/functions in the preceding cell, you are **NOT allowed to import anything else for the following purposes**:
>     - **As a part of your final solution.**
>     - **Temporarily import something to assist you to get a solution.**
>
>     **Rule of thumb:** Each part has its particular purpose to intentionally test you something. Do not attempt to find a shortcut to circumvent the rule.

In [ ]:
"""
DO NOT MAKE ANY CHANGE IN THIS CELL.
This cell generates the dataset for all subsequent parts.
"""
n, d_raw = 200, 3
w_true = np.array([2.0, -1.5, 0.5])
b_true = 3.0

X_raw = np.random.randn(n, d_raw)                        # (200, 3)
noise = 0.5 * np.random.randn(n)                         # (200,)
y = X_raw @ w_true + b_true + noise                      # (200,)

# Bias trick: prepend column of ones
X = np.hstack([np.ones((n, 1)), X_raw])                  # (200, 4)
d = X.shape[1]                                            # 4

print(f"X shape: {X.shape}")                              # (200, 4)
print(f"y shape: {y.shape}")                              # (200,)
print(f"True weights (bias, w1, w2, w3): [{b_true}, {', '.join(map(str, w_true))}]")

---

## Part 1 (10 points, non-coding task)

**Derive the normal equation** from first principles.

Starting from the MSE loss $\mathcal{L}(w) = \frac{1}{n}(Xw - y)^T(Xw - y)$:

1. Expand the quadratic form into three terms.
2. Compute $\nabla_w \mathcal{L}$ using matrix calculus.
3. Set the gradient to zero and solve for $\hat{w}$.

You may use: $\nabla_w(w^TAw) = 2Aw$ for symmetric $A$, and $\nabla_w(b^Tw) = b$.

*Reasoning is required.*

### WRITE YOUR SOLUTION HERE ###

""" END OF THIS PART """

---

## Part 2 (15 points, coding task)

**Implement the normal equation** to solve linear regression.

Given $X \in \mathbb{R}^{n \times d}$ and $y \in \mathbb{R}^n$, return $\hat{w} = (X^TX)^{-1}X^Ty$.

**Requirements**:
- Use `np.linalg.solve` instead of explicitly computing the inverse.
- No loops.

*Reasoning is not required.*

In [ ]:
def normal_equation(X: np.ndarray, y: np.ndarray) -> np.ndarray:
    """
    Solve linear regression via the normal equation.

    Args:
        X: (n, d) design matrix (includes bias column)
        y: (n,) target vector

    Returns:
        w: (d,) weight vector
    """
    ### WRITE YOUR SOLUTION HERE ###

    pass

In [ ]:
"""
DO NOT MAKE ANY CHANGE IN THIS CELL.
"""
w_normal = normal_equation(X, y)
print(f"Estimated weights: {w_normal.round(4)}")
print(f"True weights:      [{b_true}, {', '.join(map(str, w_true))}]")
assert w_normal.shape == (d,), f"Expected shape ({d},), got {w_normal.shape}"
assert np.allclose(w_normal, np.linalg.lstsq(X, y, rcond=None)[0], atol=1e-8)
print("Part 2 passed.")

""" END OF THIS PART """

---

## Part 3 (15 points, coding task)

**Implement the MSE loss and its gradient.**

- MSE: $\;\mathcal{L}(w) = \frac{1}{n}\|Xw - y\|^2$
- Gradient: $\;\nabla_w \mathcal{L} = \frac{2}{n}X^T(Xw - y)$

Both must be fully vectorized (no loops). Add shape annotations as comments.

*Reasoning is not required.*

In [ ]:
def mse_loss(X: np.ndarray, y: np.ndarray, w: np.ndarray) -> float:
    """
    Compute MSE loss: (1/n) ||Xw - y||^2

    Args:
        X: (n, d), y: (n,), w: (d,)
    Returns:
        Scalar loss value
    """
    ### WRITE YOUR SOLUTION HERE ###

    pass


def mse_gradient(X: np.ndarray, y: np.ndarray, w: np.ndarray) -> np.ndarray:
    """
    Compute gradient of MSE loss w.r.t. w: (2/n) X^T(Xw - y)

    Args:
        X: (n, d), y: (n,), w: (d,)
    Returns:
        grad: (d,) gradient vector
    """
    ### WRITE YOUR SOLUTION HERE ###

    pass

In [ ]:
"""
DO NOT MAKE ANY CHANGE IN THIS CELL.
"""
w_test = np.zeros(d)
loss_val = mse_loss(X, y, w_test)
grad_val = mse_gradient(X, y, w_test)
print(f"Loss at w=0: {loss_val:.4f}")
print(f"Gradient at w=0: {grad_val.round(4)}")
assert isinstance(loss_val, (float, np.floating)), "Loss must be a scalar"
assert grad_val.shape == (d,), f"Gradient shape must be ({d},)"

# Numerical gradient check
eps = 1e-5
num_grad = np.zeros(d)
for j in range(d):
    wp = w_test.copy(); wp[j] += eps
    wm = w_test.copy(); wm[j] -= eps
    num_grad[j] = (mse_loss(X, y, wp) - mse_loss(X, y, wm)) / (2 * eps)
assert np.allclose(grad_val, num_grad, atol=1e-5), "Gradient does not match numerical gradient"
print("Part 3 passed.")

""" END OF THIS PART """

---

## Part 4 (20 points, coding task)

**Implement gradient descent** for linear regression.

Update rule: $w^{(t+1)} = w^{(t)} - \eta \cdot \nabla_w \mathcal{L}(w^{(t)})$

Initialize $w = \mathbf{0}$. Use your `mse_gradient` from Part 3. Record the loss at every step.

*Reasoning is not required.*

In [ ]:
def gradient_descent(
    X: np.ndarray,       # (n, d)
    y: np.ndarray,       # (n,)
    lr: float = 0.01,
    n_steps: int = 1000,
) -> tuple:
    """
    Gradient descent for linear regression.

    Args:
        X: (n, d) design matrix
        y: (n,) targets
        lr: learning rate
        n_steps: number of gradient descent steps

    Returns:
        w: (d,) final weight vector
        losses: list of length n_steps, loss at each step
    """
    ### WRITE YOUR SOLUTION HERE ###

    pass

In [ ]:
"""
DO NOT MAKE ANY CHANGE IN THIS CELL.
"""
w_gd, losses = gradient_descent(X, y, lr=0.01, n_steps=3000)
print(f"GD weights:       {w_gd.round(4)}")
print(f"Normal eq weights: {w_normal.round(4)}")
print(f"Final GD loss:     {losses[-1]:.6f}")
print(f"Normal eq loss:    {mse_loss(X, y, w_normal):.6f}")
assert len(losses) == 3000, "Must record loss at every step"
assert np.allclose(w_gd, w_normal, atol=0.05), "GD should converge near normal equation"
print("Part 4 passed.")

""" END OF THIS PART """

---

### Geometric Interpretation: The Hat Matrix

The **hat matrix** $H = X(X^TX)^{-1}X^T$ is the orthogonal projector onto $\text{col}(X)$. It "puts a hat on $y$": $\hat{y} = Hy$. The residual $r = y - \hat{y}$ is perpendicular to every column of $X$.

Key properties: $H^2 = H$ (idempotent), $H^T = H$ (symmetric), $\text{trace}(H) = d$.

---

## Part 5 (15 points, coding task)

**Implement the hat matrix** and verify its properties.

Return a dictionary with:
- `"H"`: the $(n, n)$ hat matrix
- `"y_hat"`: predictions via $Hy$, shape $(n,)$
- `"residual_orth"`: $\max |X^Tr|$ (should be $\approx 0$)
- `"idempotent"`: $\max |H^2 - H|$ (should be $\approx 0$)
- `"trace"`: $\text{trace}(H)$ (should equal $d$)

*Reasoning is not required.*

In [ ]:
def hat_matrix_analysis(X: np.ndarray, y: np.ndarray) -> dict:
    """
    Compute the hat matrix and verify its properties.

    Args:
        X: (n, d), y: (n,)

    Returns:
        dict with keys: "H", "y_hat", "residual_orth", "idempotent", "trace"
    """
    ### WRITE YOUR SOLUTION HERE ###

    pass

In [ ]:
"""
DO NOT MAKE ANY CHANGE IN THIS CELL.
"""
results = hat_matrix_analysis(X, y)
print(f"H shape: {results['H'].shape}")
print(f"Residual orthogonality: {results['residual_orth']:.2e}")
print(f"Idempotent error: {results['idempotent']:.2e}")
print(f"Trace(H): {results['trace']:.4f} (expected {d})")
assert results['H'].shape == (n, n)
assert results['residual_orth'] < 1e-10
assert results['idempotent'] < 1e-10
assert abs(results['trace'] - d) < 1e-10
assert np.allclose(results['y_hat'], X @ w_normal)
print("Part 5 passed.")

""" END OF THIS PART """

---

## Part 6 (10 points, non-coding task)

**Analyze learning rate stability.**

The maximum stable learning rate for gradient descent on MSE is $\eta_{\max} = \frac{2}{\lambda_{\max}(X^TX / n)}$.

1. Explain why gradient descent diverges when $\eta > \eta_{\max}$. What happens geometrically to the iterates?
2. If the eigenvalues of $X^TX/n$ are $\{0.1, 1.0, 10.0\}$, what is $\eta_{\max}$? At learning rate $\eta = \eta_{\max}/2$, how many iterations are needed for the component associated with $\lambda_{\min} = 0.1$ to reduce its error by a factor of $e^{-1}$? (Hint: convergence rate per component is $|1 - \eta\lambda_j|$.)

*Reasoning is required.*

### WRITE YOUR SOLUTION HERE ###

""" END OF THIS PART """

---

## Part 7 (15 points, coding task)

**Learning rate experiments.**

1. Compute $\eta_{\max}$ for the given dataset using `np.linalg.eigvalsh`.
2. Run `gradient_descent` with $\eta \in \{0.001,\; 0.01,\; \eta_{\max}/2,\; 0.95 \cdot \eta_{\max}\}$ for 2000 steps each.
3. Create a single plot with all four loss curves (use log scale on y-axis). Add a horizontal dashed line at the optimal loss. Include a legend.

*Reasoning is not required.*

In [ ]:
### WRITE YOUR SOLUTION HERE ###

pass

""" END OF THIS PART """